## Imports and Setup

In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os
from PIL import Image
from tensorflow.keras.applications import EfficientNetB0
from keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from keras.models import Model
from keras.optimizers import AdamW
from keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tqdm.notebook import tqdm
import numpy as np
import keras_hub
import seaborn as sns

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load paths from .env file
dataset_dir = os.getenv("DATASET_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")

# Patgs to the data splits files
train_file = os.path.join(datasplits_dir, "mm_train.tsv")
val_file = os.path.join(datasplits_dir, "mm_val.tsv")
test_file = os.path.join(datasplits_dir, "mm_test.tsv")

# Load the data splits into pandas DataFrames
train_df = pd.read_csv(train_file, sep="\t")
val_df = pd.read_csv(val_file, sep="\t")
test_df = pd.read_csv(test_file, sep="\t")

In [ ]:
train_df.shape

In [ ]:
train_df.head(1)

In [ ]:
val_df.shape

In [ ]:
val_df.head(1)

In [ ]:
test_df.shape

In [ ]:
test_df.head(1)

## Label Encoding

In [ ]:
label_encoders = {}
for col in ["mm_info", "mm_human"]:
    le = LabelEncoder()
    all_values = pd.concat([train_df[col], val_df[col], test_df[col]])
    le.fit(all_values)

    train_df[col] = le.transform(train_df[col])
    val_df[col] = le.transform(val_df[col])
    test_df[col] = le.transform(test_df[col])

    label_encoders[col] = le

## Dataset creation

In [ ]:
# Parameters
MAX_LENGTH = train_df["cleaned_tweet_text"].str.split().str.len().max()
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

text_preprocessor = keras_hub.models.RobertaTextClassifierPreprocessor.from_preset("roberta_base_en", sequence_length=MAX_LENGTH)

def process_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = efficientnet_preprocess(img)
    return img

def encode_example(text, img_path, info_label, human_label):
    text_out = text_preprocessor(text)  # returns token_ids & padding_mask
    input_word_ids = text_out["token_ids"]
    padding_mask = text_out["padding_mask"]

    image = process_image(img_path)

    outputs = {
        "input_word_ids": input_word_ids,
        "input_padding_mask": padding_mask,
        "input_image": image}

    labels = {
        "info": tf.cast(info_label, tf.float32),
        "human": tf.cast(human_label, tf.int32)
    }
    return outputs, labels

def df_to_multimodal_dataset(df, batch_size=32, shuffle=True):
    img_paths = df["image_path"].apply(lambda p: os.path.normpath(os.path.join(dataset_dir, p))).values
    texts = df["cleaned_tweet_text"].values
    infos = df["mm_info"].values
    humans = df["mm_human"].values

    ds = tf.data.Dataset.from_tensor_slices((texts, img_paths, infos, humans))
    ds = ds.map(encode_example, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=700)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
train_ds = df_to_multimodal_dataset(train_df, batch_size=BATCH_SIZE, shuffle=True)
val_ds   = df_to_multimodal_dataset(val_df, batch_size=BATCH_SIZE, shuffle=False)
test_ds  = df_to_multimodal_dataset(test_df, batch_size=BATCH_SIZE, shuffle=False)

## Model Definition

In [ ]:
def build_multimodal_classifier(text_backbone, image_backbone):
    # TEXT BRANCH
    input_word_ids = keras.Input(shape=(None,), dtype=tf.int32, name="input_word_ids")
    padding_mask = keras.Input(shape=(None,), dtype=tf.int32, name="input_padding_mask")

    text_output = text_backbone({"token_ids": input_word_ids, "padding_mask": padding_mask})
    text_features = text_output[:, 0, :]  # CLS token

    # IMAGE BRANCH
    image_input = keras.Input(shape=(*IMAGE_SIZE, 3), name="input_image")

    image_augmentation = keras.Sequential([
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2)
    ])

    augmented_image = image_augmentation(image_input)
    image_output = image_backbone(augmented_image)
    image_features = keras.layers.GlobalAveragePooling2D()(image_output)

    # FUSION
    # projections
    text_proj = keras.layers.Dense(512, name="text_projection")(text_features)
    image_proj = keras.layers.Dense(512, name="image_projection")(image_features)
    # concatenate features and compute gate
    fusion_input = keras.layers.Concatenate()([text_proj, image_proj])
    λ = keras.layers.Dense(1, activation="sigmoid", name="modality_gate")(fusion_input)
    # modality gating fusion: (1-λ) * text + λ * image
    fused = keras.layers.Add(name="fused")([
        keras.layers.Multiply()([keras.layers.Lambda(lambda x: 1 - x)(λ), text_proj]),
        keras.layers.Multiply()([λ, image_proj])
    ])

    # CLASSIFIER MLP
    x = keras.layers.BatchNormalization()(fused)
    x = keras.layers.Dense(512)(x)
    x = keras.layers.Activation("relu")(x)
    x = keras.layers.Dropout(0.6)(x)

    # OUTPUT HEADS
    info_out = keras.layers.Dense(1, activation="sigmoid", name="info")(x)
    human_out = keras.layers.Dense(train_df["mm_human"].nunique(), activation="softmax", name="human")(x)

    # FINAL MODEL
    model = keras.Model(inputs={
        "input_word_ids": input_word_ids,
        "input_padding_mask": padding_mask,
        "input_image": image_input
    }, outputs={
        "info": info_out,
        "human": human_out
    })

    return model

In [ ]:
# Define backbone models (frozen at this stage)
text_backbone = keras_hub.models.RobertaBackbone.from_preset("roberta_base_en")
text_backbone.trainable = False

image_backbone = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
image_backbone.trainable = False

# Build full multimodal model
model = build_multimodal_classifier(text_backbone, image_backbone)

# Summary
model.summary()

## Training

In [ ]:
model.compile(
    optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-5),
    loss={
        "info": "binary_crossentropy",
        "human": "sparse_categorical_crossentropy"
    },
    metrics={
        "info": "accuracy",
        "human": "accuracy"
    }
)

# Callbacks
callbacks = [
    ModelCheckpoint("best_multimodal_model.keras", save_best_only=True, monitor="val_loss"),
    EarlyStopping(patience=5, restore_best_weights=True)
]

# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

In [ ]:
def plot_training_history(history):
	for key in history.history:
		if not key.startswith("val_"):
			plt.figure()
			plt.plot(history.history[key], label="train")
			plt.plot(history.history[f"val_{key}"], label="val")
			plt.title(key)
			plt.xlabel("Epoch")
			plt.ylabel("Value")
			plt.legend()
			plt.grid(True)
			plt.show()

In [ ]:
plot_training_history(history)

## Fine-tuning

In [ ]:
# TODO

## Evaluation

In [ ]:
def evaluate_task(y_true, y_pred, task_name, class_names=None):
        print(f"\n{task_name} - Classification Report:")

        print(classification_report(y_true, y_pred, target_names=class_names))

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

        print(f"> Accuracy:  {acc:.4f}")
        print(f"> Precision: {prec:.4f}")
        print(f"> Recall:    {rec:.4f}")
        print(f"> F1-score:  {f1:.4f}")

        # Shorten class names only for "mm_human" task
        if task_name == "Multimodal Humanitarian Categories":
                display_names = [c[0].upper() for c in class_names]
        else:
                display_names = class_names

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                                xticklabels=display_names, yticklabels=display_names)
        plt.title(f"{task_name} - Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.show()

In [ ]:
def evaluate_model(model, test_ds):
    y_true_info, y_pred_info = [], []
    y_true_human, y_pred_human = [], []

    for batch in tqdm(test_ds, desc="Predicting"):
        # Extract only the inputs from the batch tuple
        inputs, labels = batch
        preds = model.predict(inputs, verbose=0)

        # Task 1: informativeness
        true_info = labels["info"].numpy() # Access labels from the extracted labels dictionary
        pred_info = (preds["info"] > 0.5).astype("int32").flatten() # Access prediction for 'info' head
        y_true_info.extend(true_info)
        y_pred_info.extend(pred_info)

        # Task 2: humanitarian classification
        true_human = labels["human"].numpy() # Access labels from the extracted labels dictionary
        pred_human = np.argmax(preds["human"], axis=-1) # Access prediction for 'human' head
        y_true_human.extend(true_human)
        y_pred_human.extend(pred_human)

    evaluate_task(
	    y_true_info,
	    y_pred_info,
	    "Multimodal Informativeness",
	    class_names=["Informative", "Not Informative"]
    )

    evaluate_task(
	    y_true_human,
	    y_pred_human,
	    "Multimodal Humanitarian Categories",
	    class_names=["affected_individuals", "infrastructure_and_utility_damage", "not_humanitarian", "other_relevant_information", "rescue_volunteering_or_donation_effort"]
    )


In [ ]:
evaluate_model(model, test_ds)